# 01 · Acquisition, SQL integration and data quality
**DSA 2050 · Option 06 · Water supply reliability monitoring**

Decision question: *Which zones (NYCHA developments) show recurring disruption or abnormal consumption?*

This notebook covers **Week 1 (preview)** and **Week 2 (acquisition, SQL, quality, cleaning)**:

1. Download the four raw sources and write the raw-data manifest
2. Preview 10–20 rows of each source (Week 1 deliverable)
3. Load everything into SQLite and run the quality queries in `sql/queries.sql`
4. Apply cleaning rules C1–C9 with before/after row counts
5. Validate the SQL join (row counts, unmatched keys)
6. Build the development-month panel and label measurement gaps
7. Export the data-quality table and a draft data dictionary

Run top to bottom with **Kernel → Restart & Run All**. Every output table is saved to `reports/`.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import water_utils as wu
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## 1. Download the raw data
`src/download_data.py` downloads each file once into `data/raw/` (skips files that already exist) and writes `data/raw_manifest.csv` with row count, size, SHA-256 hash and download time for each file. The hash proves later that the raw file was never edited.

In [2]:
from download_data import main as download_all
manifest = download_all()
manifest

[skip] water_bills.csv already exists (use force=True to re-download)
[skip] development_data_book.csv already exists (use force=True to re-download)
[skip] weather_nyc_daily.json already exists (use force=True to re-download)
[skip] complaints_311_water_system.json already exists
                                     file  rows    bytes
                 data/raw/water_bills.csv 66432 20708710
       data/raw/development_data_book.csv   346   148977
data/raw/complaints_311_water_system.json 11252  1336591
          data/raw/weather_nyc_daily.json  4534   169464


,file,source,request_url,rows,bytes,sha256,downloaded_utc
0,data/raw/water_bills.csv,A - NYCHA Water Consumption And Cost (66be-66yr),https://data.cityofnewyork.us/resource/66be-66...,66432,20708710,a5b5c2d7437632d81acf8aeab89b83e6d64057cc0c11f4...,2026-11-16T08:29:47+00:00
1,data/raw/development_data_book.csv,B - NYCHA Development Data Book (evjd-dqpz),https://data.cityofnewyork.us/resource/evjd-dq...,346,148977,1d0cca0af3403249461df08eef16316715814d1b8d1f33...,2026-11-16T08:29:47+00:00
2,data/raw/complaints_311_water_system.json,"C - 311 Water System complaints, counts by bor...",https://data.cityofnewyork.us/resource/76ig-c5...,11252,1336591,4d1f3d678044784b4ac675eaa6e642c380dde4c117628c...,2026-11-16T08:29:47+00:00
3,data/raw/weather_nyc_daily.json,"D - Open-Meteo historical daily weather, Centr...",https://archive-api.open-meteo.com/v1/archive?...,4534,169464,c08d5fe5f26ad6be99d9f2877c54d29e6608235486a72f...,2026-11-16T08:29:47+00:00


## 2. Load the raw files and preview them
Nothing is cleaned yet. We only read the files and look at them. The 15-row previews are saved to `data/sample/` for the Week 1 submission.

In [3]:
raw = wu.load_raw()
for name, df in raw.items():
    print(f"{name:16s} rows={len(df):>9,}  columns={df.shape[1]}")

sample_dir = ROOT / "data" / "sample"
sample_dir.mkdir(exist_ok=True)
for name, df in raw.items():
    df.head(15).to_csv(sample_dir / f"preview_{name}.csv", index=False)

raw["bills_raw"].head(15)

bills_raw        rows=   66,432  columns=25
devbook_raw      rows=      346  columns=52
complaints_raw   rows=   11,252  columns=4
weather_daily    rows=    4,534  columns=6


,development_name,borough,account_name,location,meter_amr,meter_scope,tds,edp,rc_code,funding_source,amp,vendor_name,umis_bill_id,revenue_month,service_start_date,service_end_date,days,meter_number,estimated,current_charges,rate_class,bill_analyzed,consumption_hcf,water_sewer_charges,other_charges
0,FHA REPOSSESSED HOUSES (GROUP IX),FHA,FHA REPOSSESSED HOUSES (GROUP IX),353 BERRIMAN STREET,NONE,NaN,283,376,Q028300,FEDERAL,NY005012090P,NEW YORK CITY WATER BOARD,15346978,2025-05,2025-04-21T00:00:00.000,2025-05-20T00:00:00.000,29,N51521740,N,267.85,MWW W CF,Yes,55,267.85,0
1,FHA REPOSSESSED HOUSES (GROUP V),FHA,FHA REPOSSESSED HOUSES (GROUP V),124-18 135TH PLACE,AMR,NaN,260,397,Q026000,FEDERAL,NY005012090P,NEW YORK CITY WATER BOARD,15345199,2025-05,2025-04-21T00:00:00.000,2025-05-20T00:00:00.000,29,V48019927-Sewer,NaN,22.59,MWW SWR,Yes,0,22.59,0
2,UPACA (SITE 5),MANHATTAN,UPACA (SITE 5),BLD 01,AMR,NaN,343,757,M034300,FEDERAL,NY005012410P,NEW YORK CITY WATER BOARD,15346441,2025-05,2025-04-21T00:00:00.000,2025-05-19T00:00:00.000,28,FIRE,NaN,4.06,Fire,Yes,0,0,4.06
3,WSUR (BROWNSTONES),MANHATTAN,WEST SIDE URBAN RENEWAL (BROWNSTONES),NaN,NaN,NaN,178,279,M017800,FEDERAL,NY005011270P,NEW YORK CITY WATER BOARD,15343727,2025-05,2025-04-21T00:00:00.000,2025-05-20T00:00:00.000,29,G17004062-Sewer,NaN,751.1,MWW SWR,Yes,0,751.1,0
4,FHA REPOSSESSED HOUSES (GROUP V),FHA,FHA REPOSSESSED HOUSES (GROUP V),124-18 135TH PLACE,AMR,NaN,260,397,Q026000,FEDERAL,NY005012090P,NEW YORK CITY WATER BOARD,15345199,2025-05,2025-04-21T00:00:00.000,2025-05-20T00:00:00.000,29,V48019927,N,14.21,MWW W CF,Yes,0,14.21,0
5,FHA REPOSSESSED HOUSES (GROUP V),FHA,FHA REPOSSESSED HOUSES (GROUP V),198-15 116TH AVENUE,AMR,NaN,260,397,Q026000,FEDERAL,NY005012090P,NEW YORK CITY WATER BOARD,15419933,2025-05,2025-04-29T00:00:00.000,2025-05-29T00:00:00.000,30,E20235589-Sewer,N,23.37,MWW SWR,Yes,0,23.37,0
6,SOUTH JAMAICA II,QUEENS,SOUTH JAMAICA II,BLD 14,AMR,BLD 14,66,220,Q006600,FEDERAL,NY005010080P,NEW YORK CITY WATER BOARD,15346970,2025-05,2025-04-21T00:00:00.000,2025-05-20T00:00:00.000,29,K16792339,N,988.61,MWW W daCF,Yes,203,988.61,0
7,SOUTH JAMAICA II,QUEENS,SOUTH JAMAICA II,BLD 14,AMR,BLD 14,66,220,Q006600,FEDERAL,NY005010080P,NEW YORK CITY WATER BOARD,15346970,2025-05,2025-04-21T00:00:00.000,2025-05-20T00:00:00.000,29,K16792339-Sewer,NaN,1571.89,MWW SWR,Yes,0,1571.89,0
8,FHA REPOSSESSED HOUSES (GROUP V),FHA,FHA REPOSSESSED HOUSES (GROUP V),198-15 116TH AVENUE,AMR,NaN,260,397,Q026000,FEDERAL,NY005012090P,NEW YORK CITY WATER BOARD,15419933,2025-05,2025-04-29T00:00:00.000,2025-05-29T00:00:00.000,30,E20235589,N,14.7,MWW W CF,Yes,3,14.7,0
9,EAST 165TH STREET-BRYANT AVENUE,BRONX,EAST 165TH STREET-BRYANT AVENUE,NaN,NaN,NaN,304,552,B030400,FEDERAL,NY005015300P,NEW YORK CITY WATER BOARD,15415209,2025-05,2025-04-29T00:00:00.000,2025-05-29T00:00:00.000,30,O78610285-Sewer,N,2330.73,MWW SWR,Yes,0,2330.73,0


In [4]:
raw["devbook_raw"].head(5)

,data_as_of,development,hud_amp,tds,consolidated_tds,development_edp,operating_edp,hud,program,method,type,number_of_section_8_transition_apartments,number_of_current_apartments,total_number_of_apartments,number_of_rental_rooms,avg_no_r_r_per_apartment,population_section_8_transition,population_public_housing,total_population,total_of_fixed_income_household,...,density,development_cost,per_rental_room,avg_monthly_gross_rent,location_street_a,location_street_b,location_street_c,location_street_d,borough,community_distirct,us_congressional_district,ny_state_senate_district,ny_state_assembly_district,ny_city_council_district,completion_date,federalized_development,senior_development,electricity_paid_by_residents,private_management,rad_transferred_date
0,1/1/2025,1010 EAST 178TH STREET,NY005011330,180,180,289,289,NY005090,FEDERAL,CONVENTIONAL,NEW CONST,0,218,220,918.0,4.21,0,391,391,98,...,197,"$5,031,405","$5,341",$513,E TREMONT AVE,BRYANT AVE,E 178TH ST,BOSTON RD,BRONX,6,15,32,87,15,1971-03-31T00:00:00.000,NaN,NaN,NaN,NaN,2025-06-24T00:00:00.000
1,1/1/2025,104-14 TAPSCOTT STREET,NY005011670,242,167,361,283,NY005174,FEDERAL,TURNKEY,NEW CONST,NaN,28,30,122.0,4.36,0,55,55,6,...,236,"$839,110","$6,505",$604,TAPSCOTT ST,UNION ST,SUTTER AVE,BLAKE AVE,BROOKLYN,16,09,25,55,41,1972-10-31T00:00:00.000,NaN,NaN,NaN,NaN,2023-11-28T00:00:00.000
2,1/1/2025,1162-1176 WASHINGTON AVENUE,NY005013080,233,308,354,344,NY005138,FEDERAL,TURNKEY,REHAB,0,64,66,290.0,4.53,0,128,128,18,...,292,"$2,205,187","$7,302",$619,E 167TH ST,E 168TH ST,THIRD AVE,WASHINGTON AVE,BRONX,3,15,32,79,16,1975-12-31T00:00:00.000,NaN,NaN,NaN,NaN,NaN
3,1/1/2025,131 SAINT NICHOLAS AVENUE,NY005010970,154,097,264,261,NY005065,FEDERAL,CONVENTIONAL,NEW CONST,0,98,100,386.0,3.94,0,144,144,44,...,207,"$1,880,013","$4,694",$630,SAINT NICHOLAS AVE,W 116TH ST,W 117TH ST,NaN,MANHATTAN,10,13,30,70,09,1965-03-31T00:00:00.000,NaN,NaN,NaN,NaN,NaN
4,1/1/2025,1471 WATSON AVENUE,NY005010670,214,067,332,222,NY005162,FEDERAL,TURNKEY,NEW CONST,0,95,96,381.5,4.02,0,120,120,41,...,124,"$2,278,928","$5,814",$549,WATSON AVE,COLGATE AVE,EVERGREEN AVE,NaN,BRONX,9,14,32,85,17,1970-12-31T00:00:00.000,NaN,NaN,NaN,NaN,NaN


In [5]:
display(raw["complaints_raw"].head(10))
display(raw["weather_daily"].head(5))

,borough,month,descriptor,n
0,BROOKLYN,2014-03-01T00:00:00.000,Dirty Water (WE),92
1,QUEENS,2015-06-01T00:00:00.000,Excessive Water In Basement (WEFB),25
2,STATEN ISLAND,2019-07-01T00:00:00.000,Possible Water Main Break (Use Comments) (WA1),16
3,BROOKLYN,2019-08-01T00:00:00.000,No Water (WNW),63
4,MANHATTAN,2018-12-01T00:00:00.000,Other Water Problem (Use Comments) (WZZ),26
5,BRONX,2014-09-01T00:00:00.000,Dirty Water (WE),25
6,MANHATTAN,2014-12-01T00:00:00.000,Hydrant Running (WC3),66
7,STATEN ISLAND,2018-10-01T00:00:00.000,Hydrant Leaking (WC1),24
8,MANHATTAN,2015-09-01T00:00:00.000,Leak (Use Comments) (WA2),130
9,BROOKLYN,2016-08-01T00:00:00.000,Other Water Problem (Use Comments) (WZZ),37


,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum
0,2013-01-01,0.4,2.9,-5.7,0.0,0.0
1,2013-01-02,-4.8,-1.0,-7.7,0.0,0.0
2,2013-01-03,-5.1,-0.8,-9.5,0.0,0.0
3,2013-01-04,-1.4,2.1,-4.5,0.0,0.0
4,2013-01-05,0.0,3.9,-2.1,0.0,0.0


## 3. Standardise the Development Data Book
The Data Book's column names can change between yearly editions, so `standardise_devbook` searches for the columns we need and **prints what it matched**. Check the mapping below. If a field matched the wrong column, edit the mapping in `src/water_utils.py`.

In [6]:
developments, db_info = wu.standardise_devbook(raw["devbook_raw"])
for k, v in db_info.items():
    print(f"{k:24s} {v}")
developments.head()

column_mapping           {'tds': 'tds', 'development': 'development', 'borough': 'borough', 'apartments': 'total_number_of_apartments', 'population': 'total_population', 'res_buildings': 'number_of_residential_bldgs'}
rows_in                  346
rows_without_tds         5
duplicate_tds_dropped    0
rows_out                 341


,tds,development,borough,apartments,population,res_buildings
0,180,1010 EAST 178TH STREET,BRONX,220.0,391.0,1.0
1,242,104-14 TAPSCOTT STREET,BROOKLYN,30.0,55.0,1.0
2,233,1162-1176 WASHINGTON AVENUE,BRONX,66.0,128.0,1.0
3,154,131 SAINT NICHOLAS AVENUE,MANHATTAN,100.0,144.0,1.0
4,214,1471 WATSON AVENUE,BRONX,96.0,120.0,1.0


## 4. Load all sources into SQLite
SQLite is a database stored in one file (`data/processed/water.db`). No server, no password. Tables:
`bills_raw`, `developments`, `complaints_raw`, `weather_daily`.

In [7]:
con = wu.connect()
Q = wu.load_queries()
loaded = wu.write_tables(con, {
    "bills_raw": raw["bills_raw"],
    "developments": developments,
    "complaints_raw": raw["complaints_raw"],
    "weather_daily": raw["weather_daily"],
})
loaded

,table,rows,columns
0,bills_raw,66432,25
1,developments,341,6
2,complaints_raw,11252,4
3,weather_daily,4534,6


In [8]:
wu.run(con, "q01_row_counts", Q)

,table_name,n_rows
0,bills_raw,66432
1,developments,341
2,complaints_raw,11252
3,weather_daily,4534


## 5. Quality audit with SQL (before cleaning)
Each query is in `sql/queries.sql`. Read the comment above each block there to explain it in your defence.

In [9]:
print("Q2 bills by borough"); display(wu.run(con, "q02_bills_by_borough", Q))
print("Q4 missing values");   display(wu.run(con, "q04_missingness", Q))
print("Q6 type/range checks"); display(wu.run(con, "q06_type_and_range_checks", Q))

Q2 bills by borough


,borough,bills,developments,meters,pct_estimated,total_hcf
0,FHA,28375,11,404,17.6,336264.0
1,BROOKLYN,13100,91,270,16.4,172929305.0
2,MANHATTAN,11494,99,229,11.5,167412910.0
3,BRONX,6943,80,146,17.8,135764174.0
4,QUEENS,6156,21,124,21.9,49321492.0
5,STATEN ISLAND,264,9,3,0.8,14807432.0
6,NON DEVELOPMENT FACILITY,100,1,2,2.0,1013.0


Q4 missing values


,total_rows,missing_tds,missing_month,missing_consumption,missing_days,missing_meter,missing_estimated_flag
0,66432,31,0,725,27,3003,7144


Q6 type/range checks


,negative_consumption,zero_consumption,non_positive_days,days_over_62,no_meter,sewer_or_fire_lines,non_borough_value,first_month,last_month
0,0,9897,32,4327,3241,4572,28475,2013-01,2025-05


In [10]:
dups = wu.run(con, "q05_duplicates", Q)
print("Meter-months with more than one bill (top 50 shown):")
dups.head(15)

Meter-months with more than one bill (top 50 shown):


,tds,meter_number,revenue_month,n_rows,estimated_flags
0,377,NaN,2024-06,23,NaN
1,377,NaN,2021-06,21,NaN
2,377,NaN,2022-06,21,NaN
3,377,NaN,2023-06,21,NaN
4,8,NaN,2021-06,11,NaN
5,8,NaN,2022-06,11,NaN
6,8,NaN,2023-06,11,NaN
7,8,NaN,2024-06,11,NaN
8,243,NaN,2021-06,10,NaN
9,243,NaN,2022-06,10,NaN


In [11]:
print("Q3 largest actual bills in 2024 (outlier candidates to investigate, NOT to delete):")
wu.run(con, "q03_filter_largest_actual_2024", Q)

Q3 largest actual bills in 2024 (outlier candidates to investigate, NOT to delete):


,development_name,borough,meter_number,revenue_month,days,consumption_hcf,hcf_per_day
0,SEWARD PARK EXTENSION,MANHATTAN,E14081316,2024-03,102,6448.0,63.2
1,MELTZER TOWER,MANHATTAN,9020850,2024-06,92,6310.0,68.6
2,MELTZER TOWER,MANHATTAN,9020850,2024-09,91,6125.0,67.3
3,SEWARD PARK EXTENSION,MANHATTAN,E14081316,2024-06,82,5662.0,69.0
4,MELTZER TOWER,MANHATTAN,9020850,2024-12,91,5591.0,61.4
5,SEWARD PARK EXTENSION,MANHATTAN,E14122365,2024-12,91,5525.0,60.7
6,MELTZER TOWER,MANHATTAN,9020850,2024-03,50,3847.0,76.9
7,SEWARD PARK EXTENSION,MANHATTAN,K12447340,2024-01,34,3430.0,100.9
8,MELTZER TOWER,MANHATTAN,9020850,2024-01,42,3311.0,78.8
9,SEWARD PARK EXTENSION,MANHATTAN,K12447340,2024-10,32,3280.0,102.5


In [12]:
print("Q11 311 descriptors - check that the LIKE patterns in complaints_monthly catch the right ones")
wu.run(con, "q11_descriptor_list", Q).head(30)

Q11 311 descriptors - check that the LIKE patterns in complaints_monthly catch the right ones


,descriptor,complaints
0,Leak (Use Comments) (WA2),143307
1,Hydrant Running Full (WA4),113757
2,Hydrant Running (WC3),106736
3,Dirty Water (WE),105631
4,Hydrant Leaking (WC1),66032
5,Fire Hydrant Emergency (FHE),49103
6,Possible Water Main Break (Use Comments) (WA1),38562
7,Hydrant Defective (WC2),31711
8,No Water (WNW),23313
9,Excessive Water In Basement (WEFB),16379


## 6. Cleaning rules C1–C9
Rules are applied in order; each logs rows before and after. **Extreme values are flagged, not deleted**, because abnormal consumption is exactly what we are looking for. Only impossible or unusable rows are removed.

In [13]:
bills_clean, cleaning_log = wu.clean_bills(raw["bills_raw"])
cleaning_log.to_csv(ROOT / "reports" / "cleaning_log.csv", index=False)
print(f"Raw rows: {len(raw['bills_raw']):,}  ->  clean rows: {len(bills_clean):,}")
cleaning_log

Raw rows: 66,432  ->  clean rows: 28,635


,rule,rows_before,rows_after,rows_removed,reason
0,C1 standardise types,66432,66432,0,"Convert numbers/dates, upper-case text, fill m..."
1,C2 drop exact duplicates,66432,61772,4660,Identical rows add no information and double-c...
2,C3 drop missing key fields,61772,61023,749,"No month, no development ID or no consumption ..."
3,C4 keep five boroughs,61023,34765,26258,"'FHA' repossessed houses are scattered homes, ..."
4,C5 drop accounts with no meter,34765,32267,2498,Flat-rate / multifamily / commercial accounts ...
5,C6 drop sewer-only and fire-service lines,32267,29656,2611,Sewer charge lines repeat a water meter with z...
6,C7 drop invalid billing periods,29656,28758,898,<=0 days are errors; >62 days are quarterly/ca...
7,C8 drop negative consumption,28758,28758,0,"Credits/reversals are accounting entries, not ..."
8,C9 de-duplicate meter-period bills,28758,28635,123,Same meter and period billed twice: keep actua...


In [14]:
flags = bills_clean[["is_estimated", "is_zero_read", "is_odd_period"]].mean().mul(100).round(2)
print("Share of clean bills carrying each flag (%):"); print(flags)
wu.write_tables(con, {"bills_clean": bills_clean})

Share of clean bills carrying each flag (%):
is_estimated     19.09
is_zero_read      5.13
is_odd_period     1.70
dtype: float64


,table,rows,columns
0,bills_clean,28635,30


## 7. Join validation (the documented SQL join)
`bills_clean.tds` (foreign key) → `developments.tds` (primary key). A LEFT JOIN must return **exactly** as many rows as `bills_clean`. More rows would mean duplicate keys in `developments` (fan-out). Unmatched rows are bills whose development is missing from the current Data Book.

In [15]:
jv = wu.run(con, "q07_join_validation", Q)
display(jv)
assert jv.loc[0, "rows_after_left_join"] == jv.loc[0, "rows_before_join"], "Join changed the row count - check duplicate TDS"
print("Row count preserved by the join: OK")

,rows_before_join,rows_after_left_join,matched_rows,unmatched_rows,pct_unmatched,distinct_tds_in_bills,distinct_tds_unmatched
0,28635,28635,28410,225,0.79,104,2


Row count preserved by the join: OK


In [16]:
unmatched = wu.run(con, "q08_unmatched_developments", Q)
unmatched.to_csv(ROOT / "reports" / "unmatched_developments.csv", index=False)
print(f"{len(unmatched)} developments in the bills are not in the Data Book")
unmatched.head(20)

2 developments in the bills are not in the Data Book


,tds,development_name,borough,bills,first_month,last_month
0,1630,WILLIAMSBURG,BROOKLYN,201,2021-01,2021-12
1,322,EAST 004TH STREET REHAB,MANHATTAN,24,2013-05,2014-03


In [17]:
print("Q9 developments whose meters are most often NOT read (recurring measurement gaps)")
wu.run(con, "q09_recurring_measurement_gaps", Q)

Q9 developments whose meters are most often NOT read (recurring measurement gaps)


,tds,development,borough,apartments,bills,estimated_bills,pct_estimated
0,307,CLAREMONT REHAB (GROUP 2),BRONX,107.0,289,275,95.2
1,14,INGERSOLL,BROOKLYN,1840.0,144,137,95.1
2,61,VAN DYKE I,BROOKLYN,1603.0,144,137,95.1
3,43,NOSTRAND,BROOKLYN,1148.0,55,51,92.7
4,51,OCEAN BAY APARTMENTS (OCEANSIDE),QUEENS,418.0,470,431,91.7
5,5,QUEENSBRIDGE SOUTH,QUEENS,1604.0,145,132,91.0
6,111,DREW-HAMILTON,MANHATTAN,1217.0,53,48,90.6
7,4,RED HOOK EAST,BROOKLYN,1407.0,827,629,76.1
8,353,STEBBINS AVENUE-HEWITT PLACE,BRONX,120.0,421,212,50.4
9,330,WASHINGTON HEIGHTS REHAB PHASE IV (C),MANHATTAN,32.0,218,104,47.7


## 8. Build the development-month panel
`v01_monthly_views` creates monthly weather and 311 views; `q12_development_month_panel` aggregates meters to developments and joins size, weather and 311. `q13_panel_join_check` confirms the joins did not add or lose rows. Then `label_gaps` adds missing months and labels each development-month as **actual**, **estimated**, **zero_read** or **missing_bill**, and `metering_coverage` labels each development **whole** or **partial** metering (some developments are metered only at a store or community centre, so their use per apartment is near zero).

In [18]:
wu.run(con, "v01_monthly_views", Q)
check = wu.run(con, "q13_panel_join_check", Q)
display(check)
assert check.loc[0, "dev_months"] == check.loc[0, "after_joins"]
panel = wu.run(con, "q12_development_month_panel", Q)
print(f"Panel rows (development-months with a bill): {len(panel):,}")

,dev_months,after_joins,no_weather_match,no_311_match,no_devbook_match
0,8752,8752,0,0,23


Panel rows (development-months with a bill): 8,752


In [19]:
# Developments without a Data Book match have no apartment count -> kept in the panel but
# excluded from per-apartment statistics. Report how many.
print("Development-months without apartments:", panel["apartments"].isna().sum())
labeled = wu.label_gaps(panel)
labeled["metering"] = wu.metering_coverage(labeled)
labeled.drop(columns="month").to_csv(wu.PROCESSED / "panel_labeled.csv", index=False)
cov = (labeled.drop_duplicates("tds").groupby("metering").agg(developments=("tds", "size"),
        examples=("development", lambda s: ", ".join(s.dropna().head(4)))))
print("Metering coverage (partial = meter covers only a store/community centre; median < 25 gal/apt/day):")
display(cov)
print(f"Labelled panel rows incl. missing months: {len(labeled):,}")
labeled["gap_type"].value_counts().to_frame("development_months").assign(
    pct=lambda d: (100 * d["development_months"] / d["development_months"].sum()).round(1))

Development-months without apartments: 23
Metering coverage (partial = meter covers only a store/community centre; median < 25 gal/apt/day):


,developments,examples
metering,,
partial,29,"FIRST HOUSES, WILLIAMSBURG, RED HOOK EAST, QUE..."
unknown,5,"WSUR (SITE A) 120 WEST 94TH STREET, BRYANT AVE..."
whole,70,"SOUTH JAMAICA I, NOSTRAND, SOUTH JAMAICA II, B..."


Labelled panel rows incl. missing months: 9,609


,development_months,pct
gap_type,,
actual,6396,66.6
estimated,2155,22.4
missing_bill,857,8.9
zero_read,201,2.1


## 9. Data-quality table and draft data dictionary (Week 2 deliverables)

In [20]:
q6 = wu.run(con, "q06_type_and_range_checks", Q).iloc[0]
q4 = wu.run(con, "q04_missingness", Q).iloc[0]
lg = cleaning_log.set_index(cleaning_log["rule"].str.split().str[0])["rows_removed"]
dq = pd.DataFrame([
    ["Missing TDS / month / consumption", f"{q4.missing_tds:,} / {q4.missing_month:,} / {q4.missing_consumption:,}", f"C3: drop ({lg['C3']:,} rows)"],
    ["Exact duplicate rows", f"{lg['C2']:,}", "C2: drop"],
    ["Same meter-period billed twice", f"{lg['C9']:,}", "C9: keep actual read"],
    ["Borough not one of 5 boroughs (FHA, non-development)", f"{q6.non_borough_value:,}", f"C4: exclude ({lg['C4']:,} rows after C1-C3)"],
    ["No meter / un-metered accounts", f"{q6.no_meter:,}", f"C5: exclude ({lg['C5']:,} rows)"],
    ["Sewer-only and fire-service lines", f"{q6.sewer_or_fire_lines:,}", f"C6: exclude ({lg['C6']:,} rows)"],
    ["Billing days <= 0 or > 62", f"{q6.non_positive_days + q6.days_over_62:,}", f"C7: drop ({lg['C7']:,} rows)"],
    ["Negative consumption", f"{q6.negative_consumption:,}", f"C8: drop ({lg['C8']:,} rows)"],
    ["Zero consumption (clean bills)", f"{bills_clean.is_zero_read.sum():,}", "Keep + flag (gap or real shut-off)"],
    ["Estimated reads (clean bills)", f"{bills_clean.is_estimated.sum():,}", "Keep + flag; excluded from model target"],
    ["Bills not matching Data Book (join)", f"{jv.loc[0,'unmatched_rows']:,} rows / {jv.loc[0,'distinct_tds_unmatched']} developments", "Keep; no per-apartment rate"],
    ["Partial metering (store/community-centre meter only)", f"{(labeled.drop_duplicates('tds').metering == 'partial').sum()} developments", "Keep; excluded from per-apartment comparisons and Rule A"],
    ["Extreme high consumption", "see Q3", "Keep + investigate (signal, not error)"],
], columns=["Check", "Result", "Action"])
dq.to_csv(ROOT / "reports" / "data_quality_table.csv", index=False)
dq

,Check,Result,Action
0,Missing TDS / month / consumption,31 / 0 / 725,C3: drop (749 rows)
1,Exact duplicate rows,"4,660",C2: drop
2,Same meter-period billed twice,123,C9: keep actual read
3,"Borough not one of 5 boroughs (FHA, non-develo...","28,475","C4: exclude (26,258 rows after C1-C3)"
4,No meter / un-metered accounts,"3,241","C5: exclude (2,498 rows)"
5,Sewer-only and fire-service lines,"4,572","C6: exclude (2,611 rows)"
6,Billing days <= 0 or > 62,"4,359",C7: drop (898 rows)
7,Negative consumption,0,C8: drop (0 rows)
8,Zero consumption (clean bills),"1,469",Keep + flag (gap or real shut-off)
9,Estimated reads (clean bills),"5,467",Keep + flag; excluded from model target


In [21]:
def dictionary(df, table, notes):
    rows = []
    for c in df.columns:
        s = df[c]
        rows.append({"table": table, "column": c, "dtype": str(s.dtype),
                     "non_null_%": round(100 * s.notna().mean(), 1),
                     "example": s.dropna().iloc[0] if s.notna().any() else None,
                     "description": notes.get(c, "")})
    return pd.DataFrame(rows)

notes = {
    "tds": "Unique NYCHA development ID (PK in developments, FK in bills)",
    "revenue_month": "Bill month YYYY-MM", "borough": "Borough (zone level 2)",
    "n_meters": "Meters billed that month", "n_est_meters": "Meters with an estimated (not read) bill",
    "hcf": "Total consumption, hundred cubic feet", "hcf_per_day": "Sum over meters of HCF / billing days",
    "charges": "Water & sewer charges, USD", "apartments": "Total apartments (Data Book)",
    "share_estimated": "n_est_meters / n_meters", "gap_type": "actual / estimated / zero_read / missing_bill",
    "gal_per_apt_day": "hcf_per_day x 748 / apartments", "metering": "whole / partial / unknown meter coverage of the development", "complaints_total": "All 311 Water System complaints in borough-month", "c_supply": "311 supply complaints (no water, low pressure, main break), borough-month",
    "temp_mean": "Mean daily temperature, deg C", "precip_mm": "Monthly precipitation, mm",
}
dd = dictionary(labeled.drop(columns="month"), "panel_labeled", notes)
dd.to_csv(ROOT / "docs" / "data_dictionary_generated.csv", index=False)
dd

,table,column,dtype,non_null_%,example,description
0,panel_labeled,tds,int64,100.0,1,Unique NYCHA development ID (PK in development...
1,panel_labeled,revenue_month,str,100.0,2021-01,Bill month YYYY-MM
2,panel_labeled,borough,str,100.0,MANHATTAN,Borough (zone level 2)
3,panel_labeled,n_meters,float64,91.1,1.0,Meters billed that month
4,panel_labeled,n_est_meters,float64,91.1,0.0,Meters with an estimated (not read) bill
5,panel_labeled,n_zero_meters,float64,91.1,0.0,
6,panel_labeled,hcf,float64,91.1,14.7,"Total consumption, hundred cubic feet"
7,panel_labeled,hcf_per_day,float64,91.1,0.49,Sum over meters of HCF / billing days
8,panel_labeled,charges,float64,91.1,38.07,"Water & sewer charges, USD"
9,panel_labeled,development,str,99.8,FIRST HOUSES,


## 10. What to write in `docs/decisions_log.md`
For every rule above, write one line: **what** you did, **why**, **how many rows** it affected (from `reports/cleaning_log.csv`), and **what could go wrong**. Also explain the unmatched developments (Q8) and the descriptor list you accepted for 311 (Q11).

In [22]:
con.close()